In [1]:
!mkdir input_csvs output_csvs

In [2]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
from scipy.special import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch # <--- Import PyTorch

# --- Define Device ---
# Check if CUDA (GPU support) is available and set the device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}") # Good to confirm which device is used

# Load the RoBERTa model and tokenizer once
MODEL = 'Cloudy1225/stackoverflow-roberta-base-sentiment'
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

# --- Move Model to Device ---
model.to(device)
model.eval() # Set the model to evaluation mode (good practice for inference)


def extract_text_from_html(html_content):
    """Extract text from HTML content."""
    try:
        soup = BeautifulSoup(html_content, "html.parser")
        text = soup.get_text(separator=" ", strip=True)
        text = re.sub(r'\s+', ' ', text)  # Remove extra whitespace
        return text
    except Exception as e:
        print(f"Error extracting text: {e}")
        return None

def preprocess(text):
    """Preprocess text (username and link placeholders)."""
    new_text = []
    for t in text.split(' '):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return ' '.join(new_text).strip()

def analyze_sentiment(text):
    """Analyze sentiment using RoBERTa-based model with chunking for long texts."""
    try:
        text = extract_text_from_html(str(text))  # Clean HTML

        if not text:
            return None, None, None, None

        text = preprocess(text)  # Preprocess text
        # Consider using 512 as the standard max length for RoBERTa base
        max_token_length = 512
        tokens = tokenizer.tokenize(text)

        # Store results on CPU to avoid GPU memory accumulation if averaging later
        all_scores_list = []

        # Disable gradient calculations for inference (saves memory and computation)
        with torch.no_grad():
            # If the text is too long, split it into chunks based on tokens
            # Use overlap potentially for better context (optional, adds complexity)
            # Using a slightly smaller length for chunking ensures space for special tokens
            effective_chunk_length = max_token_length - 2 # Room for [CLS], [SEP]
            if len(tokens) > effective_chunk_length:
                # More robust chunking using token IDs might be better, but this works
                chunks = [tokens[i:i + effective_chunk_length] for i in range(0, len(tokens), effective_chunk_length)]

                for chunk in chunks:
                    chunk_text = tokenizer.convert_tokens_to_string(chunk)
                    # Tokenize the chunk ensuring truncation/padding
                    encoded_input = tokenizer(chunk_text, return_tensors='pt', truncation=True, padding=True, max_length=max_token_length)

                    # --- Move Input Tensor to Device ---
                    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}

                    output = model(**encoded_input)
                    # Move scores back to CPU for numpy/softmax
                    scores = output.logits[0].detach().cpu().numpy()
                    scores = softmax(scores)
                    all_scores_list.append(scores)

                # Average the scores across chunks
                if all_scores_list:
                    avg_scores = sum(all_scores_list) / len(all_scores_list)
                    negative, neutral, positive = avg_scores[0], avg_scores[1], avg_scores[2]
                    sentiment_score = positive - negative
                else: # Should not happen with this logic, but safe check
                     return None, None, None, None

            else: # Text fits within max length
                encoded_input = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=max_token_length)

                # --- Move Input Tensor to Device ---
                encoded_input = {k: v.to(device) for k, v in encoded_input.items()}

                output = model(**encoded_input)
                # Move scores back to CPU for numpy/softmax
                scores = output.logits[0].detach().cpu().numpy() # Use logits
                scores = softmax(scores)
                negative, neutral, positive = scores[0], scores[1], scores[2]
                sentiment_score = positive - negative

        return sentiment_score, negative, neutral, positive

    except Exception as e:
        print(f"Error analyzing text: {e}")
        # Ensure consistent return type on error
        return None, None, None, None


def process_csv_files(input_dir, output_dir, text_column='text'):
    """
    Processes all CSV files in the input directory, adds sentiment columns,
    and saves the modified CSV files to the output directory.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for filename in os.listdir(input_dir):
        if filename.endswith('.csv'):
            input_filepath = os.path.join(input_dir, filename)
            output_filepath = os.path.join(output_dir, filename)
            print(f"Processing {filename}...")

            try:
                df = pd.read_csv(input_filepath)

                if text_column not in df.columns:
                    print(f"Warning: Column '{text_column}' not found in {filename}. Skipping.")
                    continue

                # Fill NaN/missing values in the text column to avoid errors
                df[text_column] = df[text_column].fillna('')

                # Apply RoBERTa-based sentiment analysis
                # Note: df.apply can be slow. Consider batching for large files.
                results = df[text_column].apply(analyze_sentiment)

                # More robust unpacking in case analyze_sentiment returns None tuple
                df['sentiment_score'] = [res[0] if res else None for res in results]
                df['negative_weight'] = [res[1] if res else None for res in results]
                df['neutral_weight']  = [res[2] if res else None for res in results]
                df['positive_weight'] = [res[3] if res else None for res in results]

                df.to_csv(output_filepath, index=False)
                print(f"Processed and saved: {output_filepath}")

            except Exception as e:
                print(f"Error processing {filename}: {e}")

if __name__ == "__main__":
    # Make sure these directories exist or are created relative to your Colab env
    input_directory = "input_csvs"
    output_directory = "output_csvs_roberta"
    text_column_name = "content" # Make sure this matches your CSV column name

    # Example: Create dummy input if needed for testing
    # if not os.path.exists(input_directory):
    #     os.makedirs(input_directory)
    #     dummy_data = {'id': [1, 2, 3], 'content': ['<p>This is great!</p>', 'This is bad @user http://example.com', '<html><body>Just neutral text.</body></html>']}
    #     pd.DataFrame(dummy_data).to_csv(os.path.join(input_directory, 'test.csv'), index=False)


    process_csv_files(input_directory, output_directory, text_column_name)
    print("Processing complete.")

Using device: cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/963 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Processing processed_updated_authors_chrome os_posts_with_comments_answers (1).csv...


<ipython-input-2-2fc79b279783>:27: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  soup = BeautifulSoup(html_content, "html.parser")


Processed and saved: output_csvs_roberta/processed_updated_authors_chrome os_posts_with_comments_answers (1).csv
Processing processed_updated_authors_amazon aurora_posts_with_comments_answers.csv...
Processed and saved: output_csvs_roberta/processed_updated_authors_amazon aurora_posts_with_comments_answers.csv
Processing processed_updated_authors_alpine linux_posts_with_comments_answers.csv...
Processed and saved: output_csvs_roberta/processed_updated_authors_alpine linux_posts_with_comments_answers.csv
Processing processed_updated_authors_aws rds_posts_with_comments_answers.csv...
Processed and saved: output_csvs_roberta/processed_updated_authors_aws rds_posts_with_comments_answers.csv
Processing processed_updated_authors_alphafive_posts_with_comments_answers.csv...
Processed and saved: output_csvs_roberta/processed_updated_authors_alphafive_posts_with_comments_answers.csv
Processing processed_updated_authors_cockroachdb_posts_with_comments_answers.csv...
Processed and saved: output_c

<ipython-input-2-2fc79b279783>:27: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a filename than HTML or XML.

If you meant to use Beautiful Soup to parse the contents of a file on disk, then something has gone wrong. You should open the file first, using code like this:

    filehandle = open(your filename)

You can then feed the open filehandle into Beautiful Soup instead of using the filename.

However, if you want to parse some data that happens to look like a filename, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  soup = BeautifulSoup(html_content, "html.parser")


Processed and saved: output_csvs_roberta/processed_updated_authors_actordb_posts_with_comments_answers.csv
Processing processed_updated_authors_citusdb_posts_with_comments_answers.csv...
Processed and saved: output_csvs_roberta/processed_updated_authors_citusdb_posts_with_comments_answers.csv
Processing complete.
